# First-order hydrolysis: final CatBoost model (38 features)

This notebook implements the final first-order hydrolysis model using 38 features and fixed CatBoost hyperparameters. Data are provided in Supporting Information B.xlsx. Prepare the corresponding training feature matrix and update the input file path before running.

# Library import


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor

PACKAGES = ["numpy", "pandas", "scikit-learn", "joblib", "openpyxl"]
PACKAGES.append("catboost")
print({name: version(name) for name in PACKAGES})


# Functions


In [ ]:
def read_matrix(path, features, target=None):
    """Read an existing partition; require complete, finite numeric inputs."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Input file not found: {path}. Set the input paths to your prepared "
            "feature matrices. Study data are provided in Supporting Information B.xlsx."
        )
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path, sheet_name=0, engine="openpyxl")
    else:
        raise ValueError("Use a CSV or XLSX file (data on the first worksheet).")
    if df.empty:
        raise ValueError(f"Empty input table: {path}")
    if target is None:
        target = df.columns[0]
    if target in features:
        raise ValueError("The target column must not also be an input feature.")
    missing = [col for col in [target, *features] if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")
    X = df.loc[:, features].apply(pd.to_numeric, errors="raise")
    y = pd.to_numeric(df[target], errors="raise")
    if len(y) < 2:
        raise ValueError("At least two observations are required for R2 evaluation.")
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError(f"Missing or infinite feature values in {path.name}.")
    if not np.isfinite(y.to_numpy(dtype=float)).all():
        raise ValueError(f"Missing or infinite target values in {path.name}.")
    return X, y, target


def evaluate(model, X, y, partition):
    predictions = model.predict(X)
    metrics = {
        "Dataset": partition,
        "N": len(y),
        "R2": r2_score(y, predictions),
        "RMSE": np.sqrt(mean_squared_error(y, predictions)),
        "MAE": mean_absolute_error(y, predictions),
    }
    results = pd.DataFrame({
        "Dataset": partition,
        "Input_Row": np.arange(len(y)) + 2,
        "Observed": y.to_numpy(),
        "Predicted": predictions,
        "Residual": y.to_numpy() - predictions,
    })
    return metrics, results


# Final features and hyperparameters

In [ ]:
FEATURES = [
    'pH',
    'GATS2e',
    'AATSC5e',
    'AATSC0s',
    'GATS2c',
    'AATS1i',
    'AATSC2s',
    'GATS4v',
    'AATSC0i',
    'mindssC',
    'AATSC2e',
    'KRFP161',
    'maxHAvin',
    'T/K',
    'PubchemFP444',
    'AD2D111',
    'SubFP83',
    'AD2D106',
    'AATSC4v',
    'ASP-2',
    'ETA_EtaP_L',
    'GATS3p',
    'hmin',
    'minHBa',
    'CrippenLogP',
    'ATSC0i',
    'ATS0i',
    'IC2',
    'ATSC2s',
    'BIC1',
    'ATSC0p',
    'MLFER_BH',
    'SubFP52',
    'SdssC',
    'AATSC1p',
    'ETA_EtaP_F',
    'MATS2s',
    'GATS3s',
]

PARAMETERS = {
    'iterations': 1500,
    'learning_rate': 0.08,
    'depth': 7,
    'l2_leaf_reg': 5,
    'rsm': 0.6,
    'loss_function': 'RMSE',
    'border_count': 64,
    'random_seed': 42,
    'verbose': 0,
    'random_strength': 0.5,
    'eval_metric': 'R2',
    'bagging_temperature': 0.5,
    'allow_writing_files': False,
    'bootstrap_type': 'Bayesian',
}


# Data loading and processing

All input features are Z-score standardized based on the training set, and the corresponding transformation is applied to the validation and test sets.


In [ ]:
# Input tables must contain unstandardized numeric features.
TRAIN_FILE = Path('train_set.xlsx')
VALIDATION_FILE = None  # Optional: Path('val_set.xlsx')
TEST_FILE = None        # Optional: Path('test_set.xlsx')

TARGET_COLUMN = None  # None uses the first column, as in the original notebooks.
SAVE_OUTPUTS = True
OUTPUT_DIR = Path("results_kh_final")
MODEL_FILENAME = 'Final_CatBoost_38features.pkl'


In [ ]:
X_train, y_train, target = read_matrix(TRAIN_FILE, FEATURES, TARGET_COLUMN)
partitions = {"Train": (X_train, y_train)}
for name, path in [("Validation", VALIDATION_FILE), ("Test", TEST_FILE)]:
    if path is not None:
        X, y, _ = read_matrix(path, FEATURES, target)
        partitions[name] = (X, y)

# Standardize all input features using training-set parameters.
scaler = StandardScaler().fit(X_train)
partitions = {
    name: (
        pd.DataFrame(scaler.transform(X), columns=FEATURES, index=X.index), y
    )
    for name, (X, y) in partitions.items()
}

X_train, y_train = partitions["Train"]
print(f"Target: {target}; features: {len(FEATURES)}")
for name, (X, y) in partitions.items():
    print(f"{name}: {len(y)} observations")


# Modeling


In [ ]:
model = CatBoostRegressor(**PARAMETERS)
model.fit(X_train, y_train)


# Model evaluation

In [ ]:
metric_rows, prediction_tables = [], []
for name, (X, y) in partitions.items():
    metrics, predictions = evaluate(model, X, y, name)
    metric_rows.append(metrics)
    prediction_tables.append(predictions)

performance = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_tables, ignore_index=True)
print(performance.to_string(index=False))


# Save results

Save the fitted model and evaluation results.


In [ ]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    joblib.dump(model, OUTPUT_DIR / MODEL_FILENAME)
    performance.to_csv(OUTPUT_DIR / "performance.csv", index=False)

    print(f"Saved model and evaluation results to: {OUTPUT_DIR}")
